In [1]:
import os
import sys
sys.path.append(os.path.abspath(os.path.join(os.getcwd(), '..')))

import pandas as pd

from common.utils import DataPreprocessor, FeatureEngineer, set_seed
from common.exp_data_utils import ExperimentDataPreprocessor
from common.eval import Evaluator

MOVIELENS_DATA_DIR = "../datasets/hetrec2011-movielens-2k-v2/user_ratedmovies.dat"
RANDOM_SEED = 8

# Initialize data processors
set_seed(RANDOM_SEED)
data_preprocessor = DataPreprocessor()
experiment_data_preprocessor = ExperimentDataPreprocessor()
evaluator = Evaluator()


/home/adam/R11_Bai/DPRecSys/.venv/lib/python3.11/site-packages/transformers/utils/generic.py:441: FutureWarning: `torch.utils._pytree._register_pytree_node` is deprecated. Please use `torch.utils._pytree.register_pytree_node` instead.
  _torch_pytree._register_pytree_node(
Seed set to 8
Seed set to 42


random seed set to 8
numpy seed set to 8
torch seed set to 8
lightning seed set to 8
torch set to use deterministic algorithms


In [2]:
import joblib
import torch
import torch.nn.functional as F

model_name = "ngcf"
dir = f"artifacts/{model_name}"
prefix = f"{model_name}_k_all"
# eval_df = joblib.load(os.path.join(dir, f"{prefix}_eval_df.pkl"))
# user_dps_df = joblib.load(os.path.join(dir, f"user_dps_df.pkl"))
feature_engineer: FeatureEngineer = joblib.load(os.path.join(dir, f"feature_engineer.pkl"))

# Load embeddings
embeddings_dir = f"embeddings/{model_name}/"
user_emb: torch.Tensor = torch.load(f"{embeddings_dir}user_emb.pt")
item_emb: torch.Tensor = torch.load(f"{embeddings_dir}item_emb.pt")

print(user_emb.size())
print(item_emb.size())

torch.Size([2065, 256])
torch.Size([8397, 256])


In [3]:
interaction_df = data_preprocessor.load_and_process_df(
    file_dir=MOVIELENS_DATA_DIR,
    year_range=(2006, 2008),
)

interaction_info_df = data_preprocessor.join_item_features(
    df=interaction_df, actor_k=5, threshold=5,
)
interaction_info_df.head()

Data count: 855598
Data count after filtering by year (2006, 2008): 480608
Num of distinct users: 2103
Num of distinct items: 9519
done!
------------------------------
Filtering by min user/item interactions (10/0):
Data count before: 480608
Data count after: 480448
done!
------------------------------
==== Final Data Info: ====
Data Year Range: (2006, 2008)
Rating Threshold: 4.0
Num of interactions: 480448
Num of distinct users: 2064
Num of distinct items: 9519
extracting item features...
merging features...
interaction data count before merging: 480448
interaction data count after merging: 478404
done!


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,3,1.0,29,10,2006,23,17,16,2006-10-29 23:17:16,0,"[jack_lemmon, walter_matthau, annmargret, burg...",USA,donald_petrie,Donald Petrie,"[Comedy, Romance, [PAD], [PAD], [PAD], [PAD], ..."
1,75,32,4.5,29,10,2006,23,23,44,2006-10-29 23:23:44,1,"[[RARE], [RARE], [RARE], [RARE], [RARE]]",USA,[RARE],Siddharth Randeria,"[Sci-Fi, Thriller, [PAD], [PAD], [PAD], [PAD],..."
2,75,110,4.0,29,10,2006,23,30,8,2006-10-29 23:30:08,1,"[mel_gibson, sophie_marceau, patrick_mcgoohan,...",USA,[RARE],Mel Gibson,"[Action, Drama, War, [PAD], [PAD], [PAD], [PAD..."
3,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
4,75,163,4.0,29,10,2006,23,29,30,2006-10-29 23:29:30,1,"[antonio_banderas, salma_hayek, 1142520-joaqui...",USA,robert_rodriguez,Robert Rodriguez,"[Action, Romance, Thriller, [PAD], [PAD], [PAD..."


In [4]:
# TODO: determine which method to use for splitting
# 1. Split by year
# 2. Stratified split by user, timestamp

train_df, valid_df, test_df = experiment_data_preprocessor.stratified_time_split(
    interaction_info_df,
    time_col="timestamp",
    train_ratio=0.64,
    val_ratio=0.16,
    test_ratio=0.20,
)

TRAIN_NUM_USERS = len(train_df["userID"].unique())
TRAIN_NUM_ITEMS = len(train_df["movieID"].unique())


Splitting data into train/valid/test by time period with ratio=(0.64 : 0.16 : 0.2):
train: 305206 (63.8%)
valid: 75578 (15.8%)
test: 97620 (20.41%)
------------------------------ 

Check target label distribution after splitting (%):
train label
0    0.547669
1    0.452331
Name: proportion, dtype: float64
valid label
0    0.606737
1    0.393263
Name: proportion, dtype: float64
test label
0    0.590914
1    0.409086
Name: proportion, dtype: float64


In [5]:
train_df

,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,actorID,country,directorID,directorName,genre
0,75,2011,2.0,29,10,2006,23,16,39,2006-10-29 23:16:39,0,"[christopher_lloyd, lea_thompson, thomas_f_wil...",USA,robert_zemeckis,Robert Zemeckis,"[Action, Adventure, Comedy, Sci-Fi, [PAD], [PA..."
1,75,420,2.0,29,10,2006,23,16,42,2006-10-29 23:16:42,0,"[eddie_murphy, judge_reinhold, hector_elizondo...",USA,john_landis,John Landis,"[Action, Comedy, Crime, Thriller, [PAD], [PAD]..."
2,75,160,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,"[[RARE], laura_linney, ernie_hudson_jr, tim_cu...",USA,frank_marshall,Frank Marshall,"[Action, Adventure, Mystery, Sci-Fi, [PAD], [P..."
3,75,1304,2.5,29,10,2006,23,16,56,2006-10-29 23:16:56,0,"[paul_newman, robert_redford, katharine_ross, ...",USA,george_roy_hill,George Roy Hill,"[Action, Comedy, Western, [PAD], [PAD], [PAD],..."
4,75,353,3.5,29,10,2006,23,17,0,2006-10-29 23:17:00,0,"[[RARE], ernie_hudson_jr, michael_wincott, dav...",USA,[RARE],Alex Proyas,"[Action, Crime, Fantasy, Thriller, [PAD], [PAD..."
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305201,71534,969,4.0,3,12,2007,2,40,41,2007-12-03 02:40:41,1,"[humphrey_bogart, katharine_hepburn, robert_mo...",UK,john_huston,John Huston,"[Adventure, Comedy, Romance, War, [PAD], [PAD]..."
305202,71534,933,5.0,3,12,2007,2,40,49,2007-12-03 02:40:49,1,"[cary_grant, grace_kelly, [RARE], 1016736-john...",USA,alfred_hitchcock,Alfred Hitchcock,"[Crime, Mystery, Romance, Thriller, [PAD], [PA..."
305203,71534,1387,4.0,3,12,2007,2,41,24,2007-12-03 02:41:24,1,"[roy_scheider, robert_shaw, narrated_by_richar...",USA,steven_spielberg,Steven Spielberg,"[Action, Horror, [PAD], [PAD], [PAD], [PAD], [..."
305204,71534,1269,4.5,3,12,2007,2,41,33,2007-12-03 02:41:33,1,"[cary_grant, [RARE], raymond_massey, [RARE], p...",USA,frank_capra,Frank Capra,"[Comedy, Mystery, Thriller, [PAD], [PAD], [PAD..."


In [6]:
encoded_df = feature_engineer.transform(train_df)
encoded_df

Transformed: Re-index user/item mapping
Transformed: Encoded idx for actorID
Transformed: Encoded idx for country
Transformed: Encoded idx for directorID
Transformed: Encoded idx for genre


,userID,movieID,rating,date_day,date_month,date_year,date_hour,date_minute,date_second,timestamp,label,directorName,actorID_idx,country_idx,directorID_idx,genre_idx
0,0,1556,2.0,29,10,2006,23,16,39,2006-10-29 23:16:39,0,Robert Zemeckis,"[443, 1384, 2127, 1, 688]",37,444,"[2, 3, 6, 17, 0, 0, 0, 0]"
1,0,361,2.0,29,10,2006,23,16,42,2006-10-29 23:16:42,0,John Landis,"[664, 1245, 903, 1, 2063]",37,258,"[2, 6, 7, 18, 0, 0, 0, 0]"
2,0,137,2.0,29,10,2006,23,16,52,2006-10-29 23:16:52,0,Frank Marshall,"[1, 1376, 726, 2140, 1]",37,162,"[2, 3, 15, 17, 0, 0, 0, 0]"
3,0,1029,2.5,29,10,2006,23,16,56,2006-10-29 23:16:56,0,George Roy Hill,"[1754, 1899, 1290, 1, 1]",37,181,"[2, 6, 20, 0, 0, 0, 0, 0]"
4,0,308,3.5,29,10,2006,23,17,0,2006-10-29 23:17:00,0,Alex Proyas,"[1, 726, 1603, 556, 1]",37,1,"[2, 7, 10, 18, 0, 0, 0, 0]"
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
305201,2063,765,4.0,3,12,2007,2,40,41,2007-12-03 02:40:41,1,John Huston,"[938, 1289, 1895, 1, 2123]",36,256,"[3, 6, 16, 19, 0, 0, 0, 0]"
305202,2063,732,5.0,3,12,2007,2,40,49,2007-12-03 02:40:49,1,Alfred Hitchcock,"[370, 855, 1, 43, 1]",37,36,"[7, 15, 16, 18, 0, 0, 0, 0]"
305203,2063,1093,4.0,3,12,2007,2,41,24,2007-12-03 02:41:24,1,Steven Spielberg,"[1939, 1902, 1663, 1445, 1655]",37,494,"[2, 12, 0, 0, 0, 0, 0, 0]"
305204,2063,995,4.5,3,12,2007,2,41,33,2007-12-03 02:41:33,1,Frank Capra,"[370, 1, 1829, 1, 1777]",37,160,"[6, 15, 18, 0, 0, 0, 0, 0]"


In [7]:
encoded_df[["userID", "movieID"]].describe()

,userID,movieID
count,305206.000000,305206.000000
mean,1011.239402,3413.014387
std,587.578858,2616.147266
min,0.000000,0.000000
25%,506.000000,1010.000000
50%,1001.000000,2710.000000
75%,1507.000000,5748.000000
max,2063.000000,8395.000000


In [ ]:
gs_score_dict = {}
user_group = encoded_df[["userID", "movieID", "rating"]].groupby("userID")
for user_id, user_df in user_group:
    ratings = torch.tensor(user_df["rating"].tolist()).unsqueeze(1) # [m, 1]
    interacted_items = user_df["movieID"].tolist() # [m]
    interacted_item_emb = item_emb[interacted_items] # [m, d]
    center = (ratings * interacted_item_emb).sum(dim=0) / ratings.sum() # [d]

    # Normalize vectors
    item_norm = F.normalize(interacted_item_emb, dim=1)  # [m, d]
    center_norm = F.normalize(center, dim=0)  # [d]

    # Compute cosine similarities (dot products after normalization)
    cos_sim = (item_norm @ center_norm)  # [m]

    # weighted avg of cosine similarities
    gs_score = (ratings.squeeze(1) * cos_sim).sum() / ratings.sum(dim=0) # scalar
    gs_score_dict[user_id] = gs_score.item()

print(gs_score_dict)


{0: 0.40108147263526917,
 1: 0.5338392853736877,
 2: 0.48449641466140747,
 3: 0.6412871479988098,
 4: 0.5113397836685181,
 5: 0.5145062804222107,
 6: 0.5413745641708374,
 7: 0.44430384039878845,
 8: 0.6756030917167664,
 9: 0.5541414022445679,
 10: 0.6054080128669739,
 11: 0.44997599720954895,
 12: 0.6219052076339722,
 13: 0.45852386951446533,
 14: 0.6307879686355591,
 15: 0.5621853470802307,
 16: 0.5520792603492737,
 17: 0.5525197982788086,
 18: 0.39040103554725647,
 19: 0.6364222168922424,
 20: 0.6245563626289368,
 21: 0.6559170484542847,
 22: 0.41889381408691406,
 23: 0.6116979718208313,
 24: 0.4747993052005768,
 25: 0.6930024027824402,
 26: 0.7191045880317688,
 27: 0.46924054622650146,
 28: 0.5073168277740479,
 29: 0.7036046981811523,
 30: 0.5861787796020508,
 31: 0.5716935396194458,
 32: 0.5705193877220154,
 33: 0.5275953412055969,
 34: 0.30872124433517456,
 35: 0.6389622092247009,
 36: 0.3924306333065033,
 37: 0.5334354639053345,
 38: 0.5397823452949524,
 39: 0.5173454880714417,
 

In [18]:
user_type = {"encoded_generalist": [], "encoded_specialist": []}
for user_id, gs_score in gs_score_dict.items():
    if gs_score <= 0.5:
        user_type["encoded_generalist"].append(user_id)
    else:
        user_type["encoded_specialist"].append(user_id)

user_type["generalist"] = [feature_engineer.idx2vocab['userID'][user_id] for user_id in user_type["encoded_generalist"]]
user_type["specialist"] = [feature_engineer.idx2vocab['userID'][user_id] for user_id in user_type["encoded_specialist"]]

print(user_type.keys())

dict_keys(['encoded_generalist', 'encoded_specialist', 'generalist', 'specialist'])


In [19]:
dir = "artifacts/"
joblib.dump(user_type, os.path.join(dir, 'user_gs_type_dict.pkl'))

['artifacts/user_gs_type_dict.pkl']

In [21]:
loaded_dict = joblib.load(os.path.join(dir, 'user_gs_type_dict.pkl'))
print(loaded_dict.keys())

dict_keys(['encoded_generalist', 'encoded_specialist', 'generalist', 'specialist'])
